# Lab 02-10 · Numerical Methods for Engineers

Companion to chapter [02-10](../../chapters/02-10-numerical-methods.md). Handout: [README.md](README.md).

Run the cells in order. Cells marked **🔮 Predict** ask you to write a prediction *before* running.

In [1]:
import math, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.linalg import lu_factor, lu_solve
sys.path.insert(0, str(Path.cwd().parents[3] / "tools"))
try:
    from plotstyle import use_style; use_style()
except ImportError:
    pass

## Part 1 — Floating point up close

**🔮 Predict:** what are `0.1 + 0.2 == 0.3`, `1 - math.cos(1e-8)`, and the float32 value of `1e8 + 1`?

In [2]:
print("0.1 + 0.2 == 0.3:", 0.1 + 0.2 == 0.3, "   difference:", 0.1 + 0.2 - 0.3)
print("exact decimal value stored for 0.1:", f"{0.1:.25f}")
print("1 - cos(1e-8) direct:", 1 - math.cos(1e-8), "   2 sin^2(x/2):", 2 * math.sin(0.5e-8)**2)
print("float32(1e8) + 1 == float32(1e8):", np.float32(1e8) + np.float32(1) == np.float32(1e8),
      "  spacing near 1e8 in float32:", np.spacing(np.float32(1e8)))
for name, dt in (("float64", np.float64), ("float32", np.float32), ("float16", np.float16)):
    fi = np.finfo(dt)
    print(f"{name}: eps = {fi.eps:.4g}, max = {fi.max:.4g}, smallest normal = {fi.tiny:.4g}")

b = 1e8
d = math.sqrt(b * b - 4)
print("small root of x^2 + 1e8 x + 1: textbook", (-b + d) / 2, "  stable", 1 / ((-b - d) / 2))

0.1 + 0.2 == 0.3: False    difference: 5.551115123125783e-17
exact decimal value stored for 0.1: 0.1000000000000000055511151
1 - cos(1e-8) direct: 0.0    2 sin^2(x/2): 5.0000000000000005e-17
float32(1e8) + 1 == float32(1e8): True   spacing near 1e8 in float32: 8.0
float64: eps = 2.22e-16, max = 1.798e+308, smallest normal = 2.225e-308
float32: eps = 1.192e-07, max = 3.403e+38, smallest normal = 1.175e-38
float16: eps = 0.0009766, max = 6.55e+04, smallest normal = 6.104e-05
small root of x^2 + 1e8 x + 1: textbook -7.450580596923828e-09   stable -1e-08


Low-precision rounding of a long sum: add 0.1 twenty thousand times in each format (exact answer 2000).

**🔮 Predict:** which format drifts most, and roughly by how much?

In [3]:
for dt in (np.float64, np.float32, np.float16):
    s = dt(0)
    step = dt(0.1)
    for _ in range(20000):
        s = dt(s + step)
    print(f"{np.dtype(dt).name}: sum of 20,000 x 0.1 = {float(s):.6f}  (exact 2000)")
print("float16 spacing at 256:", np.spacing(np.float16(256)), "-> adding 0.1 (less than half the spacing) changes nothing")

float64: sum of 20,000 x 0.1 = 2000.000000  (exact 2000)
float32: sum of 20,000 x 0.1 = 1999.658813  (exact 2000)
float16: sum of 20,000 x 0.1 = 256.000000  (exact 2000)
float16 spacing at 256: 0.25 -> adding 0.1 (less than half the spacing) changes nothing


## Part 2 — Newton's method and SPICE's diode

Example 02-10.2: 5 V, 1 kΩ, diode with $I_S = 10^{-14}$ A at 27 °C.

**🔮 Predict:** starting from 0.6 V, does plain Newton converge quickly? Where does its first step land?

In [4]:
IS, R, Vs = 1e-14, 1e3, 5.0
VT = 1.380649e-23 * 300.15 / 1.602176634e-19
f = lambda v: (Vs - v) / R - IS * (math.exp(v / VT) - 1)
fp = lambda v: -1 / R - IS / VT * math.exp(v / VT)
vcrit = VT * math.log(VT / (math.sqrt(2) * IS))

def pnjlim(vnew, vold):            # ngspice's DEVpnjlim (src/spicelib/devices/devsup.c)
    if vnew > vcrit and abs(vnew - vold) > 2 * VT:
        if vold > 0:
            arg = (vnew - vold) / VT
            return vold + VT * (2 + math.log(arg - 2)) if arg > 0 else vold - VT * (2 + math.log(2 - arg))
        return VT * math.log(vnew / VT)
    return vnew

def newton(v0, limit=False, tol=1e-12, maxit=300):
    v, seq = v0, [v0]
    for k in range(1, maxit + 1):
        vn = v - f(v) / fp(v)
        if limit:
            vn = pnjlim(vn, v)
        seq.append(vn)
        if abs(vn - v) < tol:
            return k, seq
        v = vn
    return None, seq

print(f"V_T = {VT*1e3:.3f} mV, V_crit = {vcrit:.4f} V")
for v0 in (0.7, 0.6, 0.0):
    n1, s1 = newton(v0); n2, s2 = newton(v0, limit=True)
    print(f"start {v0} V: plain {n1} iterations (first step -> {s1[1]:.4f} V); limited {n2} iterations")
print("root:", s2[-1], "V   current:", (Vs - s2[-1]) / R * 1e3, "mA")

V_T = 25.865 mV, V_crit = 0.7303 V
start 0.7 V: plain 5 iterations (first step -> 0.6938 V); limited 5 iterations
start 0.6 V: plain 33 iterations (first step -> 1.3659 V); limited 8 iterations
start 0.0 V: plain 173 iterations (first step -> 5.0000 V); limited 9 iterations
root: 0.6928878323821919 V   current: 4.307112167617808 mA


The companion model: at each iterate, the diode becomes a conductance plus a current source, and the circuit is solved as a linear one. The iterates are identical to Newton's.

In [5]:
v = 0.7
for k in range(5):
    G = IS / VT * math.exp(v / VT)
    i0 = IS * (math.exp(v / VT) - 1)
    Ieq = i0 - G * v
    v_new = (Vs / R - Ieq) / (1 / R + G)          # nodal equation of the linearized circuit
    print(f"iteration {k+1}: G = {G:.5g} S, Ieq = {Ieq:.5g} A -> v = {v_new:.12f} V   (Newton: {v - f(v)/fp(v):.12f})")
    v = v_new

iteration 1: G = 0.21923 S, Ieq = -0.14779 A -> v = 0.693777812996 V   (Newton: 0.693777812996)
iteration 2: G = 0.17235 S, Ieq = -0.11512 A -> v = 0.692902882518 V   (Newton: 0.692902882518)
iteration 3: G = 0.16662 S, Ieq = -0.11114 A -> v = 0.692887836734 V   (Newton: 0.692887836734)
iteration 4: G = 0.16652 S, Ieq = -0.11107 A -> v = 0.692887832382 V   (Newton: 0.692887832382)
iteration 5: G = 0.16652 S, Ieq = -0.11107 A -> v = 0.692887832382 V   (Newton: 0.692887832382)


## Part 3 — Order of accuracy

**🔮 Predict:** halving the step should cut the error by what factor for Euler, Heun and RK4?

In [6]:
def integrate(step, h, T=1.0):
    y = 1.0
    for _ in range(round(T / h)):
        y = step(y, h)
    return y
euler = lambda y, h: y - h * y
heun = lambda y, h: y + h / 2 * (-y - (y - h * y))
def rk4(y, h):
    k1 = -y; k2 = -(y + h/2*k1); k3 = -(y + h/2*k2); k4 = -(y + h*k3)
    return y + h / 6 * (k1 + 2*k2 + 2*k3 + k4)
for name, st in (("Euler", euler), ("Heun", heun), ("RK4", rk4)):
    e = [abs(integrate(st, h) - math.exp(-1)) for h in (0.1, 0.05, 0.025)]
    print(f"{name:5s}: errors {e[0]:.3e}, {e[1]:.3e}, {e[2]:.3e}   ratios {e[0]/e[1]:.3f}, {e[1]/e[2]:.3f}")

sol = solve_ivp(lambda t, y: -y, (0, 1), [1.0], method="RK45", rtol=1e-8, atol=1e-12)
print(f"solve_ivp RK45: {sol.t.size - 1} adaptive steps, error {abs(sol.y[0, -1] - math.exp(-1)):.2e}")

Euler: errors 1.920e-02, 9.394e-03, 4.647e-03   ratios 2.044, 2.021
Heun : errors 6.615e-04, 1.592e-04, 3.905e-05   ratios 4.156, 4.076
RK4  : errors 3.332e-07, 1.998e-08, 1.223e-09   ratios 16.682, 16.337
solve_ivp RK45: 12 adaptive steps, error 7.92e-10


## Part 4 — Stiffness

Example 02-10.5: two decaying modes with time constants 1 µs and 1 ms, integrated to 5 ms.

**🔮 Predict:** how many steps will an explicit adaptive solver (RK45) take, versus an implicit one (`Radau` or `BDF`)?

In [7]:
lam = np.array([-1e6, -1e3])
rhs = lambda t, y: lam * y
for method in ("RK45", "Radau", "BDF"):
    s = solve_ivp(rhs, (0, 5e-3), [1.0, 1.0], method=method, rtol=1e-6, atol=1e-9)
    print(f"{method:5s}: {s.t.size - 1:6d} steps, slow mode at 5 ms = {s.y[1, -1]:.6f} (exact {math.exp(-5):.6f})")

h = 50e-6
for name, R in (("forward Euler", lambda z: 1 + z), ("backward Euler", lambda z: 1 / (1 - z)),
                ("trapezoidal", lambda z: (1 + z/2) / (1 - z/2)), ("exact", lambda z: np.exp(z))):
    print(f"{name:15s}: fast-mode factor {R(h*lam[0]): .4f}, slow-mode factor {R(h*lam[1]): .6f}")

RK45 :   1554 steps, slow mode at 5 ms = 0.006738 (exact 0.006738)
Radau:    143 steps, slow mode at 5 ms = 0.006738 (exact 0.006738)
BDF  :    202 steps, slow mode at 5 ms = 0.006738 (exact 0.006738)
forward Euler  : fast-mode factor -49.0000, slow-mode factor  0.950000
backward Euler : fast-mode factor  0.0196, slow-mode factor  0.952381
trapezoidal    : fast-mode factor -0.9231, slow-mode factor  0.951220
exact          : fast-mode factor  0.0000, slow-mode factor  0.951229


## Part 5 — Conditioning and mixed-precision refinement

**🔮 Predict:** for the 2×2 system of Example 02-10.6, how much does a 0.0001 change in $b_2$ move the solution?

In [8]:
A = np.array([[1.0, 1.0], [1.0, 1.0001]])
print("condition number:", np.linalg.cond(A))
print("solution for b = (2, 2.0001):", np.linalg.solve(A, [2, 2.0001]))
print("solution for b = (2, 2.0002):", np.linalg.solve(A, [2, 2.0002]))

rng = np.random.default_rng(1)
n = 200
Q1, _ = np.linalg.qr(rng.standard_normal((n, n))); Q2, _ = np.linalg.qr(rng.standard_normal((n, n)))
A = Q1 @ np.diag(np.logspace(0, -3, n)) @ Q2.T          # kappa = 1e3 by construction
x_true = rng.standard_normal(n); b = A @ x_true
lu32 = lu_factor(A.astype(np.float32))                    # factor once in single precision
x = lu_solve(lu32, b.astype(np.float32)).astype(np.float64)
print(f"kappa = {np.linalg.cond(A):.3g}")
for it in range(5):
    print(f"pass {it}: relative error {np.linalg.norm(x - x_true) / np.linalg.norm(x_true):.3e}")
    r = b - A @ x                                         # residual in double precision
    d = lu_solve(lu32, r.astype(np.float32)).astype(np.float64)
    x = x + d

condition number: 40002.00007491523
solution for b = (2, 2.0001): [1. 1.]
solution for b = (2, 2.0002): [0. 2.]


kappa = 1e+03
pass 0: relative error 3.081e-05
pass 1: relative error 4.386e-10
pass 2: relative error 3.272e-14
pass 3: relative error 2.267e-14
pass 4: relative error 2.391e-14


## Part 6 — ngspice

From the repository root, run `python tools/run_spice.py 02-10`. The netlist `diode_newton.cir` asks ngspice for the operating point of Example 02-10.2. It reports 0.6929 V and 4.307 mA, and its `rusage totiter` line shows how many Newton iterations it needed. Compare that with your plain and limited Newton runs in Part 2.

## Your turn

1. In Part 2, replace the diode by two identical diodes in series. Predict the new voltage per diode and the iteration count, then check.
2. In Part 4, change the slow time constant to 1 s and integrate for 5 s. How does RK45's step count change, and why does Radau's barely change?
3. In Part 5, factor in float16 instead of float32 (NumPy can store it, but you will need to factor with your own elimination code). Predict whether refinement still converges at κ = 10³.